# Chapter 4 Walkthrough: Market Sizing and Patient Populations

This notebook follows the chapter one block at a time, using the same code path. Run
`uv run python ch04_market/scripts/answer_key.py` once first to create the answer key.
All patient records are synthetic; the prevalence and population anchors are public
reference values used for teaching.

In [1]:
import os
from pathlib import Path
import pandas as pd

# Resolve the repository root so relative data paths work from any launch directory.
repo = Path.cwd()
while not (repo / "ch03_data").exists() and repo != repo.parent:
    repo = repo.parent
os.chdir(repo)

DATA = "ch03_data/output_data/generated_data"
LAUNCH_CODES = ["E11.9", "E11.65", "E11.40"]   # the launch condition in ICD-10
DX_COLS = [f"diagnosis_{i}" for i in range(1, 11)]   # wide diagnosis columns
PRODUCT = "Roventra"

patients = pd.read_csv(f"{DATA}/reference/patients.csv")
# payer_id lives in patient_enrollments. Join it to patients for access-rule merges.
enroll = pd.read_csv(f"{DATA}/reference/patient_enrollments.csv")
patients = patients.merge(
    enroll[["patient_id", "payer_id"]].drop_duplicates("patient_id"),
    on="patient_id", how="left",
)
answer_key = pd.read_csv("ch04_market/output_data/answer_key.csv")

# Use the mature snapshot (full claims, not early/partial) for phenotyping.
mc = pd.read_csv(f"{DATA}/claims_medical/medical_claims_mature.csv")
dx_mask = mc[DX_COLS].isin(LAUNCH_CODES).any(axis=1)
coded = mc[dx_mask]
paid_dx_count = coded.groupby("patient_id").size()   # mature claims are all paid

rx = pd.read_csv(f"{DATA}/claims_pharmacy/pharmacy_claims.csv",
                 dtype={"ndc": str, "ndc_prescribed": str})
ndc_ref = pd.read_csv(f"{DATA}/reference/ndc_codes.csv", dtype={"ndc": str})
# Join on ndc_prescribed for stable product attribution (dispensed may be a pack-size variant).
rx["drug_name"] = rx["ndc_prescribed"].map(ndc_ref.set_index("ndc")["drug_name"])
roventra = rx[rx.drug_name.eq(PRODUCT)].copy()
roventra["net"] = roventra["transaction_type"].map({"PAID": 1, "REVERSED": -1}).fillna(0)
net_fills = roventra.groupby("patient_id").net.sum()

p = patients.copy()
p["true_condition"] = p.patient_id.map(answer_key.set_index("patient_id").true_launch_condition).fillna(False)
p["coded"] = p.patient_id.isin(coded.patient_id)
p["diagnosed"] = p.patient_id.map(paid_dx_count).fillna(0).ge(1)
p["diagnosed_2plus"] = p.patient_id.map(paid_dx_count).fillna(0).ge(2)
p["age_eligible"] = p.age_band.isin(["35-49", "50-64", "65+"])
p["treated"] = p.patient_id.map(net_fills).fillna(0).gt(0)
p["untreated"] = ~p.treated

hcp_targets = pd.read_csv(f"{DATA}/reference/hcp_targets.csv")
hcp_to_account = hcp_targets.set_index("npi")["account_id"]
provider_events = pd.concat([
    mc[["patient_id", "rendering_npi"]].rename(columns={"rendering_npi": "npi"}),
    rx[["patient_id", "prescriber_npi"]].rename(columns={"prescriber_npi": "npi"}),
], ignore_index=True).dropna()
provider_events["account_id"] = provider_events.npi.map(hcp_to_account)
provider_events = provider_events.dropna(subset=["account_id"])
account_counts = (provider_events.groupby(["patient_id", "account_id"]).size()
                  .reset_index(name="n_events")
                  .sort_values(["patient_id", "n_events", "account_id"],
                               ascending=[True, False, True]))
p["account_id"] = p.patient_id.map(
    account_counts.drop_duplicates("patient_id").set_index("patient_id").account_id
)
print("patient rows:", len(p))

patient rows: 20000


## 4.1 One disease, five market sizes

In [2]:
sizes = pd.DataFrame([
    ("True condition (answer key)", p.true_condition.sum()),
    ("Launch diagnosis coded",      p.coded.sum()),
    ("Paid-claims phenotype",       p.diagnosed.sum()),
    ("Age-eligible diagnosed",      (p.diagnosed & p.age_eligible).sum()),
    ("Untreated opportunity",       (p.diagnosed & p.age_eligible & p.untreated).sum()),
], columns=["market_size", "patients"])
print(sizes.to_string(index=False))

                market_size  patients
True condition (answer key)      9308
     Launch diagnosis coded      8213
      Paid-claims phenotype      8213
     Age-eligible diagnosed      6998
      Untreated opportunity      2278


## 4.2 What claims can see, and the invisible patient

In [3]:
gates = pd.DataFrame([
    ("True condition (answer key)",               p.true_condition.sum()),
    ("... coded with a launch diagnosis",         (p.true_condition & p.coded).sum()),
    ("... found by the paid-claims phenotype",    (p.true_condition & p.diagnosed).sum()),
    ("False positives (other condition, flagged)",(~p.true_condition & p.diagnosed).sum()),
], columns=["gate", "patients"])
print(gates.to_string(index=False))

                                      gate  patients
               True condition (answer key)      9308
         ... coded with a launch diagnosis      8058
    ... found by the paid-claims phenotype      8058
False positives (other condition, flagged)       155


In [4]:
invisible = p[p.true_condition & ~p.diagnosed & ~p.treated].sort_values("patient_id")
pid = invisible.iloc[0].patient_id
print("first invisible true patient:", pid)
print(mc.loc[mc.patient_id.eq(pid), ["claim_date"] + DX_COLS[:3]].to_string(index=False))
print(rx.loc[rx.patient_id.eq(pid),
             ["date_of_service", "drug_name", "transaction_type", "reject_code"]].to_string(index=False))

first invisible true patient: PAT00046
claim_date diagnosis_1 diagnosis_2 diagnosis_3
2024-06-28       F41.9     J45.909       E78.5
date_of_service drug_name transaction_type  reject_code
     2024-06-09    Vexpro           PENDED         70.0
     2024-06-13    Vexpro             PAID          NaN
     2024-07-06    Vexpro             PAID          NaN


## 4.3 Validate the phenotype against truth

In [5]:
def diagnostics(flag, truth):
    tp = int((flag & truth).sum())
    fp = int((flag & ~truth).sum())
    fn = int((~flag & truth).sum())
    return dict(true_positive=tp, false_positive=fp, false_negative=fn,
               sensitivity=round(tp / (tp + fn), 3), precision=round(tp / (tp + fp), 3))

pd.DataFrame([
    {"rule": "1+ paid diagnosis",  **diagnostics(p.diagnosed,       p.true_condition)},
    {"rule": "2+ paid diagnoses",  **diagnostics(p.diagnosed_2plus, p.true_condition)},
])

,rule,true_positive,false_positive,false_negative,sensitivity,precision
0,1+ paid diagnosis,8058,155,1250,0.866,0.981
1,2+ paid diagnoses,6008,0,3300,0.645,1.000


## 4.4 Anchor the denominator outside the panel

In [6]:
US_ADULTS = 258_554_106
PREVALENCE = 0.113
panel_share = p.diagnosed.mean()
print(f"panel diagnosis share:      {panel_share:.3f}")
print(f"naive extrapolation:        {panel_share * US_ADULTS:,.0f}")
print(f"external prevalence anchor:  {PREVALENCE * US_ADULTS:,.0f}")

REGION_POP = {"Northeast": 57_609_148, "South": 126_018_935, "Midwest": 68_985_454, "West": 78_588_572}
national_target = round(PREVALENCE * US_ADULTS)
region_total = sum(REGION_POP.values())
targets = {r: round(national_target * pop / region_total) for r, pop in REGION_POP.items()}
targets["South"] += national_target - sum(targets.values())
observed = p[p.diagnosed].groupby("region").size()
weights = {r: targets[r] / observed[r] for r in targets}
p["population_weight"] = 0.0
p.loc[p.diagnosed, "population_weight"] = p.loc[p.diagnosed, "region"].map(weights)
pd.DataFrame([{"region": r, "target": targets[r], "observed": int(observed[r]),
               "weight": round(targets[r] / observed[r], 1)} for r in REGION_POP])

panel diagnosis share:      0.411
naive extrapolation:        106,175,244
external prevalence anchor:  29,216,614


,region,target,observed,weight
0,Northeast,5081925,1945,2612.8
1,South,11116615,2405,4622.3
2,Midwest,6085473,2073,2935.6
3,West,6932601,1790,3873.0


## 4.5 Build the opportunity funnel

In [7]:
ACCESS_PROBABILITY = {"Covered": 0.90, "Covered with PA": 0.65, "Non-covered": 0.10}
access = pd.read_csv(f"{DATA}/market_access/market_access_rules.csv",
                     parse_dates=["effective_start", "effective_end"])
ANALYSIS_DATE = pd.Timestamp("2024-12-31")
rules = access[access.product_name.eq(PRODUCT)
               & access.effective_start.le(ANALYSIS_DATE)
               & access.effective_end.ge(ANALYSIS_DATE)]
p = p.merge(rules[["payer_id", "region", "coverage_status"]], on=["payer_id", "region"], how="left")
p["access_probability"] = p.coverage_status.map(ACCESS_PROBABILITY).fillna(0.0)

elig = p.diagnosed & p.age_eligible
untr = elig & p.untreated
reachable = (p.loc[untr, "population_weight"] * p.loc[untr, "access_probability"]).sum()
funnel = pd.DataFrame([
    ("Diagnosed population",        p.loc[p.diagnosed, "population_weight"].sum(), "count"),
    ("Age eligible",               p.loc[elig, "population_weight"].sum(),        "count"),
    ("Untreated opportunity",      p.loc[untr, "population_weight"].sum(),        "count"),
    ("Reachable (access-adjusted)", reachable,                                    "expected"),
    ("Expected starts (25%)",       reachable * 0.25,                             "expected"),
], columns=["stage", "population", "type"])
funnel["population"] = funnel["population"].map(lambda v: f"{v:,.0f}")
print(funnel.to_string(index=False))

                      stage population     type
       Diagnosed population 29,216,614    count
               Age eligible 24,895,896    count
      Untreated opportunity  8,110,331    count
Reachable (access-adjusted)  3,420,603 expected
      Expected starts (25%)    855,151 expected


## 4.6 Estimate the population you cannot see

In [8]:
def chapman(source_a, source_b):
    n1, n2 = len(source_a), len(source_b)
    m = len(source_a & source_b)
    n_hat = (n1 + 1) * (n2 + 1) / (m + 1) - 1
    var = (n1 + 1) * (n2 + 1) * (n1 - m) * (n2 - m) / ((m + 1) ** 2 * (m + 2))
    return n1, n2, m, n_hat, 1.96 * var ** 0.5

paid_dx = set(coded.patient_id)
paid_rx = rx[rx.transaction_type.eq("PAID")]
class_drugs = ["Roventra", "Nexoral", "Vexpro"]
for label, source_b in [
    ("contaminated (drug class)",     set(paid_rx[paid_rx.drug_name.isin(class_drugs)].patient_id)),
    ("condition-specific (Roventra)", set(paid_rx[paid_rx.drug_name.eq(PRODUCT)].patient_id)),
]:
    n1, n2, m, n_hat, ci = chapman(paid_dx, source_b)
    print(f"{label:<30} n1={n1} n2={n2} m={m}  Chapman={n_hat:,.0f}  CI=[{n_hat-ci:,.0f}, {n_hat+ci:,.0f}]")
print("truth (answer key):", int(p.true_condition.sum()))

contaminated (drug class)      n1=8213 n2=13633 m=6943  Chapman=16,127  CI=[16,022, 16,231]
condition-specific (Roventra)  n1=8213 n2=6401 m=5541  Chapman=9,488  CI=[9,435, 9,540]
truth (answer key): 9308


## 4.7 From a count to a list: patient finding

In [9]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

lab = pd.read_csv(f"{DATA}/claims_lab/lab_results.csv")
a1c = lab[lab.test_name.eq("Hemoglobin A1c")]
diabetes_rx_patients = rx[
    rx.diagnosis_code.str.startswith("E11", na=False)
    & rx.transaction_type.eq("PAID")
].patient_id

f = p[["patient_id", "age_band", "region", "sex", "diagnosed", "true_condition"]].copy()
f["n_class_fills"] = f.patient_id.map(
    paid_rx[paid_rx.drug_name.isin(class_drugs)].groupby("patient_id").size()
).fillna(0)
f["max_a1c"] = f.patient_id.map(a1c.groupby("patient_id")["result"].max()).fillna(0)
f["has_elevated_a1c"] = (f.max_a1c >= 6.5).astype(int)
f["diabetes_rx_proxy"] = f.patient_id.isin(diabetes_rx_patients).astype(int)

X = pd.get_dummies(f.drop(columns=["patient_id", "diagnosed", "true_condition"]),
                   columns=["age_band", "region", "sex"])
Xtr, Xte, ytr, yte = train_test_split(X, f.diagnosed.astype(int), test_size=0.3,
                                      random_state=20260613, stratify=f.diagnosed)
clf = GradientBoostingClassifier(random_state=20260613).fit(Xtr, ytr)
f["score"] = clf.predict_proba(X)[:, 1]
print("held-out AUC:", round(roc_auc_score(yte, clf.predict_proba(Xte)[:, 1]), 3))

undx = f[f.diagnosed.eq(0)].sort_values("score", ascending=False)
base = undx.true_condition.mean()
print(f"undiagnosed base rate: {base:.1%}")
for frac in (0.10, 0.20):
    top = undx.head(int(len(undx) * frac))
    print(f"top {int(frac*100):>2}%: {top.true_condition.mean():.1%} true, lift {top.true_condition.mean()/base:.1f}x")
print("PAT00046 percentile among undiagnosed:",
      round(100 * (undx.score < undx.set_index("patient_id").loc["PAT00046", "score"]).mean(), 1))

held-out AUC: 0.93
undiagnosed base rate: 10.6%
top 10%: 99.9% true, lift 9.4x
top 20%: 51.3% true, lift 4.8x
PAT00046 percentile among undiagnosed: 94.4


The chapter's final deliverable is a linked set of quantities: national scale, patient
composition, scenario assumptions, the estimated unseen population, and the account
support. Keep them visible together rather than reporting one headline number.